In [ ]:
# In[1]:

import pandas as pd
import pytz

# load files into kernel variables (kept for later steps)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# timezone for display (UTC+8)
tz = pytz.timezone("Asia/Shanghai")

def prepare_timestamp_column(df, ts_col="timestamp"):
    # create timezone-aware datetime column from epoch seconds
    # keep original numeric timestamp column as-is, add timestamp_dt
    ts = pd.to_datetime(df[ts_col], unit="s", utc=True)
    df["timestamp_dt"] = ts.dt.tz_convert(tz)
    return df

metric_df = prepare_timestamp_column(metric_df)
trace_df = prepare_timestamp_column(trace_df)
log_df = prepare_timestamp_column(log_df)
error_logs_df = prepare_timestamp_column(error_logs_df)

def preview_list(vals, limit=20):
    # return up to `limit` items as a compact string
    if vals is None:
        return ""
    vals = list(pd.unique(pd.Series(vals).dropna()))
    preview = vals[:limit]
    return ", ".join(map(str, preview))

def file_summary(df, filename, name_field=None):
    rows = df.shape[0]
    cmdb_vals = pd.unique(df["cmdb_id"].dropna())
    cmdb_count = len(cmdb_vals)
    cmdb_preview = preview_list(cmdb_vals, 20)
    if name_field and name_field in df.columns:
        name_vals = pd.unique(df[name_field].dropna())
        name_count = len(name_vals)
        name_preview = preview_list(name_vals, 20)
    else:
        name_count = 0
        name_preview = ""
    # min/max timestamp in timezone-aware form (compact)
    if df["timestamp_dt"].dropna().empty:
        min_ts = None
        max_ts = None
    else:
        min_ts = df["timestamp_dt"].min().strftime("%Y-%m-%d %H:%M:%S %Z%z")
        max_ts = df["timestamp_dt"].max().strftime("%Y-%m-%d %H:%M:%S %Z%z")
    return {
        "file": filename,
        "rows": rows,
        "distinct_cmdb_count": cmdb_count,
        "cmdb_preview": cmdb_preview,
        "distinct_name_count": name_count,
        "name_preview": name_preview,
        "min_timestamp": min_ts,
        "max_timestamp": max_ts,
    }

summaries = [
    file_summary(metric_df, "metric.csv", name_field="kpi_name"),
    file_summary(trace_df, "trace.csv", name_field="trace_name"),
    file_summary(log_df, "log.csv", name_field="log_name"),
    file_summary(error_logs_df, "error_logs.csv", name_field=None),
]

summary_df = pd.DataFrame(summaries)

# keep output compact: selected columns and at most 20 rows (we have 4)
summary_df[["file","rows","distinct_cmdb_count","cmdb_preview","distinct_name_count","name_preview","min_timestamp","max_timestamp"]]

```
Out[1]:
```
```python
# Build a plain-English summary from the previously computed `summary_df`
summary_lines = []
for _, r in summary_df.iterrows():
    file = r["file"]
    rows = int(r["rows"])
    cmdb_count = int(r["distinct_cmdb_count"])
    cmdb_preview = r["cmdb_preview"] or "None"
    name_count = int(r["distinct_name_count"])
    name_preview = r["name_preview"] or "None"
    min_ts = r["min_timestamp"] or "None"
    max_ts = r["max_timestamp"] or "None"
    line = (
        f"{file}: {rows} rows; {cmdb_count} distinct cmdb_id(s) (examples: {cmdb_preview}); "
        f"{name_count} distinct metric/trace/log names (examples: {name_preview}); "
        f"time range {min_ts} to {max_ts}."
    )
    summary_lines.append(line)

summary = "Summary of loaded telemetry files (plain English):\n" + "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

file    rows  distinct_cmdb_count                                       cmdb_preview  distinct_name_count                                       name_preview                 min_timestamp                 max_timestamp
0      metric.csv  321027                   59  adservice, adservice-0, adservice-1, adservice...                 1384  app.grpc.count, app.grpc.mrt, app.grpc.rr, app...  2022-03-20 21:00:00 CST+0800  2022-03-20 22:00:00 CST+0800
1       trace.csv   32600                   40  adservice-0, adservice-1, adservice-2, adservi...                  200  trace.from_frontend-0.duration_mean, trace.fro...  2022-03-20 21:30:00 CST+0800  2022-03-20 21:59:00 CST+0800
2         log.csv    1794                   31  adservice-0, adservice-1, adservice-2, cartser...                    2                     log.error_count, log.row_count  2022-03-20 21:30:00 CST+0800  2022-03-20 22:00:00 CST+0800
3  error_logs.csv       0                    0                                                                       0                                                                             None                          None```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd
import pytz

# Reuse metric_df, ensure timestamp_dt exists and tz is Asia/Shanghai
tz = pytz.timezone("Asia/Shanghai")
if "timestamp_dt" not in metric_df.columns:
    metric_df["timestamp_dt"] = pd.to_datetime(metric_df["timestamp"], unit="s", utc=True).dt.tz_convert(tz)

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full series (before any filtering)
p95_series = metric_df.groupby(["cmdb_id", "kpi_name"])["value"].quantile(0.95)
p95_df = p95_series.reset_index().rename(columns={"value": "global_P95"})
# keep p95_df in kernel for later steps

# 2) Filter metric.csv to the incident window [2022-03-20 21:30:00, 2022-03-20 22:00:00)
start = pd.Timestamp("2022-03-20 21:30:00").tz_localize(tz)
end = pd.Timestamp("2022-03-20 22:00:00").tz_localize(tz)
metric_window_df = metric_df[(metric_df["timestamp_dt"] >= start) & (metric_df["timestamp_dt"] < end)].copy()

# 3) Merge global P95 into the windowed data and compute breaches
window_merged = metric_window_df.merge(p95_df, on=["cmdb_id", "kpi_name"], how="left")
window_merged["is_breach"] = window_merged["value"] >= window_merged["global_P95"]

# Aggregations per (cmdb_id, kpi_name)
grp = window_merged.groupby(["cmdb_id", "kpi_name"])
anomaly_count = grp["is_breach"].sum().rename("anomaly_count_in_window")
earliest_breach = (window_merged[window_merged["is_breach"]]
                   .groupby(["cmdb_id", "kpi_name"])["timestamp_dt"]
                   .min()
                   .rename("earliest_breach_timestamp"))
max_value = grp["value"].max().rename("max_value_in_window")
total_points = grp.size().rename("total_points_in_window")

# Combine into a single dataframe
anomalies_df = pd.concat([anomaly_count, earliest_breach, max_value, total_points], axis=1).reset_index()
# Merge in global_P95
anomalies_df = anomalies_df.merge(p95_df, on=["cmdb_id", "kpi_name"], how="left")

# Keep only those with at least one breach
anomalies_df = anomalies_df[anomalies_df["anomaly_count_in_window"] > 0].copy()

# Format earliest_breach_timestamp as string (compact); keep NaT as None (shouldn't occur due to filter)
anomalies_df["earliest_breach_timestamp"] = anomalies_df["earliest_breach_timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S %Z%z")

# 4) Prepare top 20 sorted by anomaly_count descending
anomalies_top20 = anomalies_df.sort_values(by="anomaly_count_in_window", ascending=False).head(20)[
    ["cmdb_id", "kpi_name", "global_P95", "anomaly_count_in_window", "earliest_breach_timestamp", "max_value_in_window", "total_points_in_window"]
].reset_index(drop=True)

total_breached_pairs = anomalies_df.shape[0]

# Display compact outputs (variables)
p95_df.shape, metric_window_df.shape, window_merged.shape, anomalies_top20, total_breached_pairs

```
Out[2]:
```
```python
# Build a plain-English compact summary using previously computed kernel variables
top_n = 5
top_rows = anomalies_top20.head(top_n)

lines = [
    f"Metric preprocessing & anomaly scan summary:",
    f"- Global P95 thresholds computed for {p95_df.shape[0]} distinct (cmdb_id, kpi_name) pairs.",
    f"- Metric rows inside incident window [2022-03-20 21:30:00, 2022-03-20 22:00:00): {metric_window_df.shape[0]} rows.",
    f"- Rows merged with global P95 in window: {window_merged.shape[0]} rows.",
    f"- Distinct (cmdb_id, kpi_name) with >=1 breach in window: {total_breached_pairs}.",
    f"- Top {min(top_n, len(top_rows))} breached (cmdb_id, kpi_name) by anomaly count:"
]

for i, r in top_rows.iterrows():
    idx = i + 1
    lines.append(
        f"  {idx}. {r['cmdb_id']} | {r['kpi_name']} — anomalies={int(r['anomaly_count_in_window'])}, "
        f"earliest_breach={r['earliest_breach_timestamp']}, max_value={r['max_value_in_window']}, "
        f"total_points={int(r['total_points_in_window'])}"
    )

lines.append(f"- A compact top-20 table is available as variable `anomalies_top20` in the kernel for inspection.")
summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

((5281, 3), (158070, 5), (158070, 7),                cmdb_id                                           kpi_name  global_P95  anomaly_count_in_window     earliest_breach_timestamp  max_value_in_window  total_points_in_window
0   shippingservice2-0  container.node-6.container_ulimits_soft.max_op...   1048576.0                       30  2022-03-20 21:30:00 CST+0800            1048576.0                      30
1   shippingservice2-0             container.node-6.container_threads_max         0.0                       30  2022-03-20 21:30:00 CST+0800                  0.0                      30
2            adservice                                        app.grpc.rr       100.0                       30  2022-03-20 21:30:00 CST+0800                100.0                      30
3            adservice                                        app.grpc.sr       100.0                       30  2022-03-20 21:30:00 CST+0800                100.0                      30
4            adservice                                     app.http.count        24.0                       30  2022-03-20 21:30:00 CST+0800                 24.0                      30
5            adservice                                        app.http.rr       100.0                       30  2022-03-20 21:30:00 CST+0800                100.0                      30
6            adservice                                        app.http.sr       100.0                       30  2022-03-20 21:30:00 CST+0800                100.0                      30
7            adservice    runtime.java_lang_ClassLoading_LoadedClassCount      5162.0                       30  2022-03-20 21:30:00 CST+0800               5162.0                      30
8            adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...      5185.0                       30  2022-03-20 21:30:00 CST+0800               5185.0                      30
9            adservice  runtime.java_lang_ClassLoading_UnloadedClassCount        23.0                       30  2022-03-20 21:30:00 CST+0800                 23.0                      30
10           adservice             runtime.java_lang_ClassLoading_Verbose         0.0                       30  2022-03-20 21:30:00 CST+0800                  0.0                      30
11           adservice  runtime.java_lang_Compilation_CompilationTimeM...         1.0                       30  2022-03-20 21:30:00 CST+0800                  1.0                      30
12  shippingservice2-0                 container.node-6.container_threads        31.0                       30  2022-03-20 21:30:00 CST+0800                 31.0                      30
13  shippingservice2-0  container.node-6.container_tasks_state.uninter...         0.0                       30  2022-03-20 21:30:00 CST+0800                  0.0                      30
14           adservice  runtime.java_lang_GarbageCollector_CollectionC...         8.0                       30  2022-03-20 21:30:00 CST+0800                  8.0                      30
15  shippingservice2-0     container.node-6.container_tasks_state.stopped         0.0                       30  2022-03-20 21:30:00 CST+0800                  0.0                      30
16  shippingservice2-0      container.node-6.container_fs_reads./dev/vda1         0.0                       30  2022-03-20 21:30:00 CST+0800                  0.0                      30
17  shippingservice2-0    container.node-6.container_fs_reads_MB./dev/vda         0.0                       30  2022-03-20 21:30:00 CST+0800                  0.0                      30
18  shippingservice2-0  container.node-6.container_fs_reads_merged./de...         0.0                       30  2022-03-20 21:30:00 CST+0800                  0.0                      30
19  shippingservice2-0  container.node-6.container_fs_sector_reads./de...         0.0                       30  2022-03-20 21:30:00 CST+0800                  0.0                      30, 4405)```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd
import pytz

# Reuse existing kernel variable window_merged
tz = pytz.timezone("Asia/Shanghai")

# 1) Filter breach rows where value >= global_P95
window_breaches = window_merged[window_merged["is_breach"] == True].copy()

# 2) If no breaches, create empty summary
if window_breaches.empty:
    component_breach_summary = pd.DataFrame(
        columns=[
            "cmdb_id",
            "distinct_breached_kpis",
            "total_anomaly_points",
            "earliest_breach_timestamp",
            "latest_breach_timestamp",
            "max_value",
            "max_globalP95",
            "severity_ratio",
        ]
    )
    total_distinct_cmdbs = 0
else:
    # Aggregations per cmdb_id
    distinct_breached_kpis = window_breaches.groupby("cmdb_id")["kpi_name"].nunique().rename("distinct_breached_kpis")
    total_anomaly_points = window_breaches.groupby("cmdb_id").size().rename("total_anomaly_points")
    earliest_breach = window_breaches.groupby("cmdb_id")["timestamp_dt"].min().rename("earliest_breach_timestamp")
    latest_breach = window_breaches.groupby("cmdb_id")["timestamp_dt"].max().rename("latest_breach_timestamp")
    max_value = window_breaches.groupby("cmdb_id")["value"].max().rename("max_value")
    # max_globalP95 across breached rows (global_P95 is per row/kpi and computed globally earlier)
    max_globalP95 = window_breaches.groupby("cmdb_id")["global_P95"].max().rename("max_globalP95")

    # Combine into a single DataFrame
    component_breach_summary = pd.concat(
        [
            distinct_breached_kpis,
            total_anomaly_points,
            earliest_breach,
            latest_breach,
            max_value,
            max_globalP95,
        ],
        axis=1,
    ).reset_index()

    # Format timestamps as strings with timezone
    component_breach_summary["earliest_breach_timestamp"] = component_breach_summary["earliest_breach_timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S %Z%z")
    component_breach_summary["latest_breach_timestamp"] = component_breach_summary["latest_breach_timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S %Z%z")

    # Severity ratio: max_value / max_globalP95 (float). Handle division by zero by using np.nan
    component_breach_summary["severity_ratio"] = component_breach_summary.apply(
        lambda r: float(r["max_value"]) / float(r["max_globalP95"]) if (pd.notna(r["max_globalP95"]) and r["max_globalP95"] != 0) else np.nan,
        axis=1,
    )

    # Sort as requested and limit to top 20 for display
    component_breach_summary = component_breach_summary.sort_values(
        by=["distinct_breached_kpis", "total_anomaly_points"], ascending=[False, False]
    ).reset_index(drop=True)

    total_distinct_cmdbs = component_breach_summary.shape[0]

# Prepare compact top-20 view
component_breach_summary_top20 = component_breach_summary.head(20)[
    [
        "cmdb_id",
        "distinct_breached_kpis",
        "total_anomaly_points",
        "earliest_breach_timestamp",
        "latest_breach_timestamp",
        "max_value",
        "max_globalP95",
        "severity_ratio",
    ]
]

# Display compact results (variables)
component_breach_summary_top20, total_distinct_cmdbs

```
Out[3]:
```
```python
# Build a plain-English summary from the existing kernel variables
# Uses: component_breach_summary_top20, total_distinct_cmdbs
top_n = 10

if "component_breach_summary_top20" not in globals() or "total_distinct_cmdbs" not in globals():
    summary = "Required kernel variables not found: component_breach_summary_top20 or total_distinct_cmdbs."
else:
    df = component_breach_summary_top20.copy()
    n_display = min(top_n, df.shape[0])
    lines = []
    lines.append("Compact summary of component-level breaches (incident window 2022-03-20 21:30 to 22:00 CST+0800):")
    lines.append(f"- Total distinct cmdb_id with ≥1 breach: {total_distinct_cmdbs}.")
    lines.append(f"- Showing top {n_display} components sorted by distinct breached KPIs then breach count:")

    for i in range(n_display):
        r = df.iloc[i]
        lines.append(
            f"{i+1}. {r['cmdb_id']} — distinct_breached_kpis={int(r['distinct_breached_kpis'])}, "
            f"total_anomaly_points={int(r['total_anomaly_points'])}, "
            f"earliest={r['earliest_breach_timestamp']}, latest={r['latest_breach_timestamp']}, "
            f"severity_ratio={r['severity_ratio']:.6g}"
        )

    lines.append("- Note: severity_ratio = max_value / max_globalP95 (float).")
    lines.append("Detailed per-component summary is available in variable `component_breach_summary`.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id  distinct_breached_kpis  total_anomaly_points     earliest_breach_timestamp       latest_breach_timestamp     max_value  max_globalP95  severity_ratio
0                 adservice                     337                  9212  2022-03-20 21:30:00 CST+0800  2022-03-20 21:59:00 CST+0800  2.054492e+12   2.054134e+12        1.000174
1                adservice2                     318                  8894  2022-03-20 21:30:00 CST+0800  2022-03-20 21:59:00 CST+0800  1.700233e+12   1.699893e+12        1.000200
2                frontend-2                     178                  3671  2022-03-20 21:30:00 CST+0800  2022-03-20 21:59:00 CST+0800  1.647785e+09   1.647785e+09        1.000000
3         checkoutservice-2                     138                  2500  2022-03-20 21:30:00 CST+0800  2022-03-20 21:59:00 CST+0800  1.647785e+09   1.647785e+09        1.000000
4                frontend-0                     136                  3682  2022-03-20 21:30:00 CST+0800  2022-03-20 21:59:00 CST+0800  1.647785e+09   1.647785e+09        1.000000
5         checkoutservice-1                     134                  2443  2022-03-20 21:30:00 CST+0800  2022-03-20 21:59:00 CST+0800  1.647785e+09   1.647785e+09        1.000000
6               frontend2-0                     124                  3637  2022-03-20 21:30:00 CST+0800  2022-03-20 21:59:00 CST+0800  1.647785e+09   1.647785e+09        1.000000
7                frontend-1                     118                  3429  2022-03-20 21:30:00 CST+0800  2022-03-20 21:59:00 CST+0800  1.647785e+09   1.647785e+09        1.000000
8         checkoutservice-0                     110                  2437  2022-03-20 21:30:00 CST+0800  2022-03-20 21:59:00 CST+0800  1.647785e+09   1.647785e+09        1.000000
9             cartservice-2                      93                  2116  2022-03-20 21:30:00 CST+0800  2022-03-20 21:59:00 CST+0800  1.647785e+09   1.647785e+09        1.000000
10  productcatalogservice-0                      92                  2389  2022-03-20 21:30:00 CST+0800  2022-03-20 21:59:00 CST+0800  1.647785e+09   1.647785e+09        1.000000
11            cartservice-0                      92                  2057  2022-03-20 21:30:00 CST+0800  2022-03-20 21:59:00 CST+0800  1.647785e+09   1.647785e+09        1.000000
12  productcatalogservice-1                      88                  2303  2022-03-20 21:30:00 CST+0800  2022-03-20 21:59:00 CST+0800  1.647785e+09   1.647785e+09        1.000000
13  productcatalogservice-2                      86                  2179  2022-03-20 21:30:00 CST+0800  2022-03-20 21:59:00 CST+0800  1.647785e+09   1.647785e+09        1.000000
14            cartservice-1                      86                  2007  2022-03-20 21:30:00 CST+0800  2022-03-20 21:59:00 CST+0800  1.647785e+09   1.647785e+09        1.000000
15              adservice-2                      85                  2165  2022-03-20 21:30:00 CST+0800  2022-03-20 21:59:00 CST+0800  1.647785e+09   1.647785e+09        1.000000
16              adservice-0                      85                  2093  2022-03-20 21:30:00 CST+0800  2022-03-20 21:59:00 CST+0800  1.647785e+09   1.647785e+09        1.000000
17       checkoutservice2-0                      82                  2326  2022-03-20 21:30:00 CST+0800  2022-03-20 21:59:00 CST+0800  1.647785e+09   1.647785e+09        1.000000
18  recommendationservice-0                      81                  2176  2022-03-20 21:30:00 CST+0800  2022-03-20 21:59:00 CST+0800  1.647785e+09   1.647785e+09        1.000000
19  recommendationservice-2                      81                  2175  2022-03-20 21:30:00 CST+0800  2022-03-20 21:59:00 CST+0800  1.647785e+09   1.647785e+09        1.000000, 59)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone("Asia/Shanghai")

# Ensure trace_df has timestamp_dt
if "timestamp_dt" not in trace_df.columns:
    trace_df["timestamp_dt"] = pd.to_datetime(trace_df["timestamp"], unit="s", utc=True).dt.tz_convert(tz)
else:
    # ensure timezone is correct
    trace_df["timestamp_dt"] = pd.to_datetime(trace_df["timestamp_dt"]).dt.tz_localize(None).dt.tz_localize("UTC").dt.tz_convert(tz)

# 1) Compute global P95 per (cmdb_id, trace_name) using the full trace series
trace_p95 = trace_df.groupby(["cmdb_id", "trace_name"])["value"].quantile(0.95).reset_index().rename(columns={"value": "global_P95"})

# 2) Filter trace.csv to the incident window [2022-03-20 21:30:00, 2022-03-20 22:00:00)
start = pd.Timestamp("2022-03-20 21:30:00").tz_localize(tz)
end = pd.Timestamp("2022-03-20 22:00:00").tz_localize(tz)
trace_window = trace_df[(trace_df["timestamp_dt"] >= start) & (trace_df["timestamp_dt"] < end)].copy()

# 3) Merge P95 into the window and mark breaches where value >= global_P95
trace_window_merged = trace_window.merge(trace_p95, on=["cmdb_id", "trace_name"], how="left")
trace_window_merged["is_breach"] = trace_window_merged["value"] >= trace_window_merged["global_P95"]

# 4) Aggregate breach rows per cmdb_id
breach_rows = trace_window_merged[trace_window_merged["is_breach"] == True].copy()

if breach_rows.empty:
    trace_component_breach_summary = pd.DataFrame(
        columns=[
            "cmdb_id",
            "distinct_breached_traces",
            "total_breach_points",
            "earliest_breach_timestamp",
            "latest_breach_timestamp",
            "max_value_in_breaches",
            "max_globalP95",
            "severity_ratio",
        ]
    )
    total_distinct_cmdbs = 0
    trace_breached_pairs_top20 = pd.DataFrame(
        columns=[
            "cmdb_id",
            "trace_name",
            "global_P95",
            "breach_count",
            "earliest_breach_timestamp",
            "max_value_in_window",
        ]
    )
else:
    grp = breach_rows.groupby("cmdb_id")
    distinct_breached_traces = grp["trace_name"].nunique().rename("distinct_breached_traces")
    total_breach_points = grp.size().rename("total_breach_points")
    earliest_breach = grp["timestamp_dt"].min().rename("earliest_breach_timestamp")
    latest_breach = grp["timestamp_dt"].max().rename("latest_breach_timestamp")
    max_value_in_breaches = grp["value"].max().rename("max_value_in_breaches")
    max_globalP95 = grp["global_P95"].max().rename("max_globalP95")

    trace_component_breach_summary = pd.concat(
        [
            distinct_breached_traces,
            total_breach_points,
            earliest_breach,
            latest_breach,
            max_value_in_breaches,
            max_globalP95,
        ],
        axis=1,
    ).reset_index()

    # format timestamps
    trace_component_breach_summary["earliest_breach_timestamp"] = trace_component_breach_summary["earliest_breach_timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S %Z%z")
    trace_component_breach_summary["latest_breach_timestamp"] = trace_component_breach_summary["latest_breach_timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S %Z%z")

    # severity ratio
    trace_component_breach_summary["severity_ratio"] = trace_component_breach_summary.apply(
        lambda r: float(r["max_value_in_breaches"]) / float(r["max_globalP95"]) if (pd.notna(r["max_globalP95"]) and r["max_globalP95"] != 0) else np.nan,
        axis=1,
    )

    # sort and keep top 20 for display
    trace_component_breach_summary = trace_component_breach_summary.sort_values(
        by=["distinct_breached_traces", "total_breach_points"], ascending=[False, False]
    ).reset_index(drop=True)

    total_distinct_cmdbs = trace_component_breach_summary.shape[0]

    # 5) Prepare top 20 (cmdb_id, trace_name) pairs sorted by breach count
    pair_grp = breach_rows.groupby(["cmdb_id", "trace_name"])
    pair_breach_count = pair_grp.size().rename("breach_count")
    pair_earliest = pair_grp["timestamp_dt"].min().rename("earliest_breach_timestamp")
    pair_max_value = pair_grp["value"].max().rename("max_value_in_window")

    trace_breached_pairs = pd.concat([pair_breach_count, pair_earliest, pair_max_value], axis=1).reset_index()
    # merge global_P95 from trace_p95
    trace_breached_pairs = trace_breached_pairs.merge(trace_p95, on=["cmdb_id", "trace_name"], how="left")
    trace_breached_pairs["earliest_breach_timestamp"] = trace_breached_pairs["earliest_breach_timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S %Z%z")

    trace_breached_pairs_top20 = trace_breached_pairs.sort_values(by="breach_count", ascending=False).head(20)[
        ["cmdb_id", "trace_name", "global_P95", "breach_count", "earliest_breach_timestamp", "max_value_in_window"]
    ].reset_index(drop=True)

# Keep variables in kernel with requested names
# trace_p95, trace_window, trace_window_merged, trace_component_breach_summary, trace_breached_pairs_top20

# Return compact outputs: top component summary (head 20), top breached pairs (top 20), and total distinct cmdb_ids with breaches
trace_component_breach_summary.head(20), trace_breached_pairs_top20, total_distinct_cmdbs

```
Out[4]:
```
```python
# Build a plain-English summary of the trace-level scan results using kernel variables
p95_pairs = int(trace_p95.shape[0]) if "trace_p95" in globals() else 0
window_rows = int(trace_window.shape[0]) if "trace_window" in globals() else 0
merged_rows = int(trace_window_merged.shape[0]) if "trace_window_merged" in globals() else 0
breached_cmdbs = int(total_distinct_cmdbs) if "total_distinct_cmdbs" in globals() else 0
breached_pairs_rows = int(trace_breached_pairs_top20.shape[0]) if "trace_breached_pairs_top20" in globals() else 0

if breached_cmdbs == 0:
    summary = (
        "Trace-level anomaly scan (incident window 2022-03-20 21:30 to 22:00 CST+0800):\n"
        f"- Global P95 thresholds computed for {p95_pairs} distinct (cmdb_id, trace_name) pairs.\n"
        f"- Trace rows inside incident window: {window_rows} rows; after merging thresholds: {merged_rows} rows.\n"
        "- No trace breaches detected in the incident window (0 components with ≥1 breached trace).\n"
        "- No (cmdb_id, trace_name) pairs had breaches in the window.\n"
        "- Intermediate tables available in kernel: trace_p95, trace_window, trace_window_merged, "
        "trace_component_breach_summary, trace_breached_pairs_top20."
    )
else:
    summary = (
        "Trace-level anomaly scan (incident window 2022-03-20 21:30 to 22:00 CST+0800):\n"
        f"- Global P95 thresholds computed for {p95_pairs} distinct (cmdb_id, trace_name) pairs.\n"
        f"- Trace rows inside incident window: {window_rows} rows; after merging thresholds: {merged_rows} rows.\n"
        f"- Distinct components with ≥1 trace breach: {breached_cmdbs}.\n"
        f"- Top breached (cmdb_id, trace_name) pairs (up to 20) are in variable trace_breached_pairs_top20 "
        f"({breached_pairs_rows} rows returned).\n"
        "- Intermediate tables available in kernel: trace_p95, trace_window, trace_window_merged, "
        "trace_component_breach_summary, trace_breached_pairs_top20."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, distinct_breached_traces, total_breach_points, earliest_breach_timestamp, latest_breach_timestamp, max_value_in_breaches, max_globalP95, severity_ratio]
Index: [], Empty DataFrame
Columns: [cmdb_id, trace_name, global_P95, breach_count, earliest_breach_timestamp, max_value_in_window]
Index: [], 0)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

tz = pytz.timezone("Asia/Shanghai")

# Ensure window_merged exists and has timestamp_dt and is_breach columns
# If window_breaches exists from previous steps reuse it; otherwise compute from window_merged
if "window_merged" not in globals():
    raise RuntimeError("Required kernel variable 'window_merged' not found.")

# compute breach rows (value >= global_P95). Prefer existing window_breaches if present.
if "window_breaches" in globals():
    breaches = window_breaches
else:
    breaches = window_merged[window_merged["is_breach"] == True].copy()

# Ensure timestamp_dt is timezone-aware in Asia/Shanghai
if "timestamp_dt" not in breaches.columns:
    breaches["timestamp_dt"] = pd.to_datetime(breaches["timestamp"], unit="s", utc=True).dt.tz_convert(tz)
else:
    # convert to timezone-aware Asia/Shanghai if needed
    if breaches["timestamp_dt"].dt.tz is None:
        breaches["timestamp_dt"] = pd.to_datetime(breaches["timestamp_dt"], utc=True).dt.tz_convert(tz)
    else:
        breaches["timestamp_dt"] = pd.to_datetime(breaches["timestamp_dt"]).dt.tz_convert(tz)

# Group by per-minute timestamp (timestamp_dt is already floored to minute in telemetry)
grp = breaches.groupby("timestamp_dt")

breach_timeseries = grp.agg(
    distinct_cmdbs_breached = ("cmdb_id", "nunique"),
    total_breach_rows = ("cmdb_id", "size"),
    distinct_kpis_breached = ("kpi_name", "nunique")
).reset_index()

# Format timestamp column as string with timezone for compact display
breach_timeseries["timestamp"] = breach_timeseries["timestamp_dt"].dt.strftime("%Y-%m-%d %H:%M:%S %Z%z")

# Reorder columns: timestamp first, then metrics
breach_timeseries = breach_timeseries[["timestamp_dt", "timestamp", "distinct_cmdbs_breached", "total_breach_rows", "distinct_kpis_breached"]]

# Store in kernel variable as requested
# breach_timeseries contains full per-minute series (with timestamp_dt for sorting/filtering)
# For compact display, prepare top 20 sorted by distinct_cmdbs_breached desc then total_breach_rows desc then timestamp asc
breach_timeseries_top20 = breach_timeseries.sort_values(
    by=["distinct_cmdbs_breached", "total_breach_rows", "timestamp_dt"],
    ascending=[False, False, True]
).head(20)[["timestamp", "distinct_cmdbs_breached", "total_breach_rows", "distinct_kpis_breached"]].reset_index(drop=True)

# Determine the two peak minutes: sort by distinct_cmdbs_breached desc, timestamp asc and take first two
peak_sorted = breach_timeseries.sort_values(by=["distinct_cmdbs_breached", "timestamp_dt"], ascending=[False, True])
peak_two_df = peak_sorted.head(2)[["timestamp", "distinct_cmdbs_breached", "total_breach_rows", "distinct_kpis_breached"]].reset_index(drop=True)
# also provide a simple list of the two timestamp strings (earliest-first in case of tie ordering above)
peak_two_timestamps = peak_two_df["timestamp"].tolist()

# Keep full timeseries in kernel variable 'breach_timeseries' (as requested)
# Display compact outputs: top20 timeseries and peak two timestamps list
breach_timeseries, breach_timeseries_top20, peak_two_timestamps

```
Out[5]:
```
```python
# Build a plain-English summary from existing kernel variables: breach_timeseries, breach_timeseries_top20, peak_two_timestamps
bt = breach_timeseries  # already in kernel
bt_top = breach_timeseries_top20
peaks = peak_two_timestamps

minutes_count = int(bt.shape[0])
distinct_cmdbs_set = bt["distinct_cmdbs_breached"].unique()
distinct_cmdbs_info = f"all minutes have {int(distinct_cmdbs_set[0])} distinct cmdb_id breached" if len(distinct_cmdbs_set) == 1 else f"distinct cmdbs vary: {distinct_cmdbs_set}"

min_total = int(bt["total_breach_rows"].min())
max_total = int(bt["total_breach_rows"].max())
min_kpis = int(bt["distinct_kpis_breached"].min())
max_kpis = int(bt["distinct_kpis_breached"].max())

# minute(s) with highest total breach rows (earliest in case of tie)
max_total_val = bt["total_breach_rows"].max()
max_total_rows = bt[bt["total_breach_rows"] == max_total_val].sort_values("timestamp_dt").iloc[0]
max_total_minute = max_total_rows["timestamp"]

# second highest by total breach rows (earliest if tie), if available
sorted_by_total = bt.sort_values(by=["total_breach_rows", "timestamp_dt"], ascending=[False, True]).reset_index(drop=True)
second_total_minute = None
if sorted_by_total.shape[0] >= 2:
    second_total_minute = sorted_by_total.iloc[1]["timestamp"]

summary = (
    f"Per-minute breach summary (2022-03-20 21:30–22:00 CST+0800):\n"
    f"- Minutes analyzed: {minutes_count}.\n"
    f"- {distinct_cmdbs_info}.\n"
    f"- Per-minute total breach rows ranged from {min_total} to {max_total} rows.\n"
    f"- Distinct breached KPIs per minute ranged from {min_kpis} to {max_kpis}.\n"
    f"- Minute with highest total breach rows: {max_total_minute} ({int(max_total_val)} rows).\n"
    f"- 2nd highest by total breach rows: {second_total_minute} ({int(sorted_by_total.iloc[1]['total_breach_rows']) if second_total_minute is not None else 'N/A'} rows).\n"
    f"- By distinct components breached, all minutes are tied (59 components); per tie-break rule the two peak minutes are: {peaks}.\n"
    f"- Full per-minute series is in kernel variable 'breach_timeseries'; top-20 view in 'breach_timeseries_top20'; peak minutes list in 'peak_two_timestamps'."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                timestamp_dt                     timestamp  distinct_cmdbs_breached  total_breach_rows  distinct_kpis_breached
0  2022-03-20 21:30:00+08:00  2022-03-20 21:30:00 CST+0800                       59               3574                     916
1  2022-03-20 21:31:00+08:00  2022-03-20 21:31:00 CST+0800                       59               3607                     948
2  2022-03-20 21:32:00+08:00  2022-03-20 21:32:00 CST+0800                       59               3558                     911
3  2022-03-20 21:33:00+08:00  2022-03-20 21:33:00 CST+0800                       59               3588                     940
4  2022-03-20 21:34:00+08:00  2022-03-20 21:34:00 CST+0800                       59               3559                     907
..                       ...                           ...                      ...                ...                     ...
25 2022-03-20 21:55:00+08:00  2022-03-20 21:55:00 CST+0800                       59               3545                     886
26 2022-03-20 21:56:00+08:00  2022-03-20 21:56:00 CST+0800                       59               3550                     891
27 2022-03-20 21:57:00+08:00  2022-03-20 21:57:00 CST+0800                       59               3640                     897
28 2022-03-20 21:58:00+08:00  2022-03-20 21:58:00 CST+0800                       59               3627                     892
29 2022-03-20 21:59:00+08:00  2022-03-20 21:59:00 CST+0800                       59               3638                     894

[30 rows x 5 columns],                        timestamp  distinct_cmdbs_breached  total_breach_rows  distinct_kpis_breached
0   2022-03-20 21:53:00 CST+0800                       59               3650                     904
1   2022-03-20 21:57:00 CST+0800                       59               3640                     897
2   2022-03-20 21:59:00 CST+0800                       59               3638                     894
3   2022-03-20 21:58:00 CST+0800                       59               3627                     892
4   2022-03-20 21:39:00 CST+0800                       59               3624                     971
5   2022-03-20 21:31:00 CST+0800                       59               3607                     948
6   2022-03-20 21:44:00 CST+0800                       59               3596                     930
7   2022-03-20 21:33:00 CST+0800                       59               3588                     940
8   2022-03-20 21:37:00 CST+0800                       59               3582                     925
9   2022-03-20 21:30:00 CST+0800                       59               3574                     916
10  2022-03-20 21:35:00 CST+0800                       59               3568                     913
11  2022-03-20 21:45:00 CST+0800                       59               3568                     904
12  2022-03-20 21:34:00 CST+0800                       59               3559                     907
13  2022-03-20 21:32:00 CST+0800                       59               3558                     911
14  2022-03-20 21:43:00 CST+0800                       59               3556                     891
15  2022-03-20 21:54:00 CST+0800                       59               3555                     894
16  2022-03-20 21:56:00 CST+0800                       59               3550                     891
17  2022-03-20 21:55:00 CST+0800                       59               3545                     886
18  2022-03-20 21:41:00 CST+0800                       59               3544                     891
19  2022-03-20 21:40:00 CST+0800                       59               3542                     884, ['2022-03-20 21:30:00 CST+0800', '2022-03-20 21:31:00 CST+0800'])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone("Asia/Shanghai")

# Ensure required kernel variables exist
if "window_merged" not in globals():
    raise RuntimeError("Required kernel variable 'window_merged' not found in kernel.")
if "peak_two_timestamps" not in globals():
    raise RuntimeError("Required kernel variable 'peak_two_timestamps' not found in kernel.")

# Ensure timestamp_dt exists and is tz-aware Asia/Shanghai
if "timestamp_dt" not in window_merged.columns:
    window_merged["timestamp_dt"] = pd.to_datetime(window_merged["timestamp"], unit="s", utc=True).dt.tz_convert(tz)
else:
    # normalize to tz
    window_merged["timestamp_dt"] = pd.to_datetime(window_merged["timestamp_dt"], utc=True).dt.tz_convert(tz)

# 1) Build pair_first_breach for pairs that had at least one breach
# Identify breach rows
breach_rows = window_merged[window_merged["is_breach"] == True].copy()

# If no breaches, create empty structures
if breach_rows.empty:
    pair_first_breach = pd.DataFrame(
        columns=[
            "cmdb_id",
            "kpi_name",
            "global_P95",
            "first_breach_timestamp",
            "first_breach_timestamp_dt",
            "max_value_in_window",
            "severity_ratio",
        ]
    )
else:
    # first breach timestamp per pair (breach_rows)
    first_ts = breach_rows.groupby(["cmdb_id", "kpi_name"])["timestamp_dt"].min().rename("first_breach_timestamp_dt")
    # global_P95 per pair (should be constant per pair, take max or first)
    global_p95 = breach_rows.groupby(["cmdb_id", "kpi_name"])["global_P95"].first().rename("global_P95")
    # max value in the incident window for that pair: compute from full window_merged (all rows in window)
    max_val_window = window_merged.groupby(["cmdb_id", "kpi_name"])["value"].max().rename("max_value_in_window")
    # combine
    pair_first_breach = pd.concat([first_ts, global_p95, max_val_window], axis=1).reset_index()
    # format timestamp string matching previous convention
    pair_first_breach["first_breach_timestamp"] = pair_first_breach["first_breach_timestamp_dt"].dt.strftime("%Y-%m-%d %H:%M:%S %Z%z")
    # severity ratio
    pair_first_breach["severity_ratio"] = pair_first_breach.apply(
        lambda r: (float(r["max_value_in_window"]) / float(r["global_P95"]))
        if (pd.notna(r["global_P95"]) and r["global_P95"] != 0)
        else np.nan,
        axis=1,
    )
    # reorder columns
    pair_first_breach = pair_first_breach[
        ["cmdb_id", "kpi_name", "global_P95", "first_breach_timestamp", "first_breach_timestamp_dt", "max_value_in_window", "severity_ratio"]
    ]

# store pair_first_breach in kernel (already assigned)

# 2) For each peak timestamp, filter pairs whose first_breach_timestamp equals that peak,
#    then aggregate per cmdb_id as requested and collect sample_top_kpis (up to 10 ordered by severity_ratio desc)
peaks = peak_two_timestamps  # expected list of two timestamp strings

peak_aggregations = []

for peak in peaks:
    # filter pairs whose first_breach_timestamp matches peak
    # Note: ensure exact string match; pair_first_breach uses same formatting
    matched = pair_first_breach[pair_first_breach["first_breach_timestamp"] == peak].copy()
    if matched.empty:
        continue
    # For sample_top_kpis ordering, sort matched by severity_ratio desc, then group
    matched_sorted = matched.sort_values(by=["cmdb_id", "severity_ratio"], ascending=[True, False])
    # aggregate per cmdb_id
    agg = matched.groupby("cmdb_id").agg(
        first_breach_kpi_count = ("kpi_name", "nunique"),
        max_severity_ratio = ("severity_ratio", "max"),
    ).reset_index()
    # attach earliest_first_breach (the peak) and sample_top_kpis
    def sample_top(group):
        # group is subset of matched for a cmdb_id
        gs = group.sort_values(by="severity_ratio", ascending=False)["kpi_name"].dropna().astype(str).unique()
        topk = gs[:10]
        return ", ".join(topk)
    samples = matched.groupby("cmdb_id").apply(sample_top).rename("sample_top_kpis").reset_index()
    agg = agg.merge(samples, on="cmdb_id", how="left")
    agg["peak_timestamp"] = peak
    # reorder columns
    agg = agg[["peak_timestamp", "cmdb_id", "first_breach_kpi_count", "max_severity_ratio", "sample_top_kpis"]]
    peak_aggregations.append(agg)

# Combine per-peak aggregations into peak_component_candidates (store in kernel)
if peak_aggregations:
    peak_component_candidates = pd.concat(peak_aggregations, axis=0).reset_index(drop=True)
    # For each peak, sort by first_breach_kpi_count desc then max_severity_ratio desc, and keep top 10 per peak
    peak_component_candidates = peak_component_candidates.sort_values(
        by=["peak_timestamp", "first_breach_kpi_count", "max_severity_ratio"],
        ascending=[True, False, False]
    )
    top_per_peak = peak_component_candidates.groupby("peak_timestamp").head(10).reset_index(drop=True)
    # final display table: top 10 per peak, combined, limit to 20 rows total
    top_candidates_display = top_per_peak.head(20)[["peak_timestamp", "cmdb_id", "first_breach_kpi_count", "max_severity_ratio", "sample_top_kpis"]]
else:
    peak_component_candidates = pd.DataFrame(columns=["peak_timestamp", "cmdb_id", "first_breach_kpi_count", "max_severity_ratio", "sample_top_kpis"])
    top_candidates_display = peak_component_candidates.copy()

# 3) Return compact results:
# - top_candidates_display (<=20 rows)
# - pair_first_breach.head(20) for inspection (compact)
pair_first_breach_head20 = pair_first_breach.head(20)

# Expose variables in kernel as requested:
# pair_first_breach, peak_component_candidates

top_candidates_display, pair_first_breach_head20, peak_component_candidates.shape

```
Out[6]:
```
```python
# Build a concise plain-English summary from existing kernel variables
# Expected kernel variables: top_candidates_display, pair_first_breach_head20, peak_component_candidates

# Defensive checks
missing = [v for v in ("top_candidates_display", "pair_first_breach_head20", "peak_component_candidates") if v not in globals()]
if missing:
    summary = f"Required kernel variables missing: {', '.join(missing)}"
else:
    pc = top_candidates_display.copy()
    pfb_head = pair_first_breach_head20.copy()
    peaks_list = pc["peak_timestamp"].unique().tolist()
    total_breached_pairs = pair_first_breach.shape[0] if "pair_first_breach" in globals() else "unknown"
    total_peak_candidates = peak_component_candidates.shape[0]

    lines = []
    lines.append(f"Summary of first-breach analysis (incident window 2022-03-20 21:30–22:00 CST+0800):")
    lines.append(f"- Total (cmdb_id, kpi_name) pairs that first-breached in the window: {total_breached_pairs}.")
    if peaks_list:
        lines.append(f"- Peak minutes analyzed: {', '.join(peaks_list)}.")
    else:
        lines.append(f"- No peak minutes available.")

    # For each peak, list top 5 candidate components (cmdb_id) with counts and severity
    for peak in peaks_list:
        sub = pc[pc["peak_timestamp"] == peak].head(10)
        if sub.empty:
            lines.append(f"- {peak}: no candidate components.")
            continue
        lines.append(f"- {peak}: top candidate components (by number of KPIs first-breaching):")
        for i, row in sub.head(5).iterrows():
            lines.append(
                f"  • {row['cmdb_id']}: first_breach_kpi_count={int(row['first_breach_kpi_count'])}, "
                f"max_severity_ratio={float(row['max_severity_ratio']):.3g}"
            )

    # Note a couple of notable findings from pair_first_breach_head20
    sample_notice = []
    if not pfb_head.empty:
        sample_notice.append(f"- Example pairs (from pair_first_breach head): {len(pfb_head)} rows shown.")
        # show the first 3 kpi examples from the head
        sample_kpis = pfb_head["kpi_name"].dropna().astype(str).unique()[:3].tolist()
        if sample_kpis:
            sample_notice.append(f"  Examples: {', '.join(sample_kpis)}.")
    else:
        sample_notice.append("- No example breached pairs to show in pair_first_breach_head20.")

    lines.extend(sample_notice)
    lines.append(f"- Total per-peak candidate rows collected: {total_peak_candidates}.")
    lines.append("- Variables available in kernel for follow-up: pair_first_breach, peak_component_candidates, top_candidates_display (compact).")

    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  peak_timestamp                  cmdb_id  first_breach_kpi_count  max_severity_ratio                                    sample_top_kpis
0   2022-03-20 21:30:00 CST+0800               adservice2                     295            1.016667  runtime.jvm_threads_state.RUNNABLE, runtime.ja...
1   2022-03-20 21:30:00 CST+0800                adservice                     292            1.200000  runtime.jvm_threads_state.TIMED_WAITING, runti...
2   2022-03-20 21:30:00 CST+0800               frontend-2                     131            1.016393  mesh.source.frontend.productcatalogservice.ist...
3   2022-03-20 21:30:00 CST+0800               frontend-0                     129            1.061444  mesh.source.frontend.shippingservice.istio_req...
4   2022-03-20 21:30:00 CST+0800              frontend2-0                     121            1.000000  container.node-6.container_fs_limit_MB./dev/vd...
5   2022-03-20 21:30:00 CST+0800               frontend-1                     114            1.000000  container.node-5.container_fs_limit_MB./dev/vd...
6   2022-03-20 21:30:00 CST+0800  productcatalogservice-0                      83            1.089274  container.node-6.container_network_transmit_MB...
7   2022-03-20 21:30:00 CST+0800        checkoutservice-2                      83            1.083333  container.node-5.container_cpu_cfs_periods, me...
8   2022-03-20 21:30:00 CST+0800        checkoutservice-0                      80            1.000000  container.node-6.container_file_descriptors, c...
9   2022-03-20 21:30:00 CST+0800        checkoutservice-1                      78            1.000000  container.node-6.container_fs_limit_MB./dev/vd...
10  2022-03-20 21:31:00 CST+0800        checkoutservice-1                      18            1.083333  mesh.source.checkoutservice.jaeger-collector.i...
11  2022-03-20 21:31:00 CST+0800                   node-1                      14           19.157556  system.io.rkb_s, system.io.util, system.io.r_s...
12  2022-03-20 21:31:00 CST+0800               frontend-2                       9            6.610526  mesh.source.frontend.checkoutservice.istio_req...
13  2022-03-20 21:31:00 CST+0800        checkoutservice-2                       7            1.228155  mesh.source.checkoutservice.currencyservice.is...
14  2022-03-20 21:31:00 CST+0800            cartservice-2                       7            1.071429  mesh.destination.checkoutservice.cartservice.i...
15  2022-03-20 21:31:00 CST+0800         paymentservice-0                       6            1.142857  mesh.destination.checkoutservice.paymentservic...
16  2022-03-20 21:31:00 CST+0800                   node-6                       6            1.039768  system.net.packets_in.count, system.net.bytes_...
17  2022-03-20 21:31:00 CST+0800                   node-5                       5            1.913043  system.io.r_await, system.net.bytes_rcvd, syst...
18  2022-03-20 21:31:00 CST+0800        currencyservice-0                       5            1.000000  mesh.destination.checkoutservice.currencyservi...
19  2022-03-20 21:31:00 CST+0800         paymentservice-2                       4           11.144185  container.node-6.container_network_receive_MB....,       cmdb_id                                           kpi_name    global_P95        first_breach_timestamp first_breach_timestamp_dt  max_value_in_window  severity_ratio
0   adservice                                       app.grpc.mrt      0.002843  2022-03-20 21:53:00 CST+0800 2022-03-20 21:53:00+08:00             0.003812        1.340912
1   adservice                                        app.grpc.rr    100.000000  2022-03-20 21:30:00 CST+0800 2022-03-20 21:30:00+08:00           100.000000        1.000000
2   adservice                                        app.grpc.sr    100.000000  2022-03-20 21:30:00 CST+0800 2022-03-20 21:30:00+08:00           100.000000        1.000000
3   adservice                                     app.http.count     24.000000  2022-03-20 21:30:00 CST+0800 2022-03-20 21:30:00+08:00            24.000000        1.000000
4   adservice                                        app.http.rr    100.000000  2022-03-20 21:30:00 CST+0800 2022-03-20 21:30:00+08:00           100.000000        1.000000
5   adservice                                        app.http.sr    100.000000  2022-03-20 21:30:00 CST+0800 2022-03-20 21:30:00+08:00           100.000000        1.000000
6   adservice    runtime.java_lang_ClassLoading_LoadedClassCount   5162.000000  2022-03-20 21:30:00 CST+0800 2022-03-20 21:30:00+08:00          5162.000000        1.000000
7   adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...   5185.000000  2022-03-20 21:30:00 CST+0800 2022-03-20 21:30:00+08:00          5185.000000        1.000000
8   adservice  runtime.java_lang_ClassLoading_UnloadedClassCount     23.000000  2022-03-20 21:30:00 CST+0800 2022-03-20 21:30:00+08:00            23.000000        1.000000
9   adservice             runtime.java_lang_ClassLoading_Verbose      0.000000  2022-03-20 21:30:00 CST+0800 2022-03-20 21:30:00+08:00             0.000000             NaN
10  adservice  runtime.java_lang_Compilation_CompilationTimeM...      1.000000  2022-03-20 21:30:00 CST+0800 2022-03-20 21:30:00+08:00             1.000000        1.000000
11  adservice  runtime.java_lang_Compilation_TotalCompilation...  47018.000000  2022-03-20 21:34:00 CST+0800 2022-03-20 21:34:00+08:00         47018.000000        1.000000
12  adservice  runtime.java_lang_GarbageCollector_CollectionC...  36860.416667  2022-03-20 21:57:00 CST+0800 2022-03-20 21:57:00+08:00         36865.833333        1.000147
13  adservice  runtime.java_lang_GarbageCollector_CollectionC...      8.000000  2022-03-20 21:30:00 CST+0800 2022-03-20 21:30:00+08:00             8.000000        1.000000
14  adservice  runtime.java_lang_GarbageCollector_CollectionT...  66926.333333  2022-03-20 21:57:00 CST+0800 2022-03-20 21:57:00+08:00         66935.666667        1.000139
15  adservice  runtime.java_lang_GarbageCollector_CollectionT...    480.000000  2022-03-20 21:30:00 CST+0800 2022-03-20 21:30:00+08:00           480.000000        1.000000
16  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...      1.000000  2022-03-20 21:30:00 CST+0800 2022-03-20 21:30:00+08:00             1.000000        1.000000
17  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...      1.000000  2022-03-20 21:30:00 CST+0800 2022-03-20 21:30:00+08:00             1.000000        1.000000
18  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...      2.000000  2022-03-20 21:30:00 CST+0800 2022-03-20 21:30:00+08:00             2.333333        1.166667
19  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     94.000000  2022-03-20 21:30:00 CST+0800 2022-03-20 21:30:00+08:00            94.000000        1.000000, (84, 5))```
```

In [ ]:
# In[7]:

import pandas as pd

# Reuse existing kernel variables: pair_first_breach, peak_two_timestamps
# Safety checks
if "pair_first_breach" not in globals():
    raise RuntimeError("Kernel variable 'pair_first_breach' not found.")
if "peak_two_timestamps" not in globals():
    raise RuntimeError("Kernel variable 'peak_two_timestamps' not found.")

peaks = list(peak_two_timestamps)  # expecting two timestamp strings

def top_pairs_for_peak(peak_str, df, limit=20):
    matched = df[df["first_breach_timestamp"] == peak_str].copy()
    if matched.empty:
        cols = ["peak_timestamp", "cmdb_id", "kpi_name", "global_P95", "first_breach_timestamp", "max_value_in_window", "severity_ratio"]
        return pd.DataFrame(columns=cols)
    # sort by severity_ratio desc, then max_value_in_window desc as tie-breaker
    matched_sorted = matched.sort_values(by=["severity_ratio", "max_value_in_window"], ascending=[False, False])
    top = matched_sorted.head(limit).copy()
    top["peak_timestamp"] = peak_str
    # select and order columns as requested
    top = top[["peak_timestamp", "cmdb_id", "kpi_name", "global_P95", "first_breach_timestamp", "max_value_in_window", "severity_ratio"]]
    # reset index for compact display
    return top.reset_index(drop=True)

# Produce peak1_top_pairs and peak2_top_pairs (if fewer peaks available, produce empty DataFrame for missing)
peak1_top_pairs = top_pairs_for_peak(peaks[0], pair_first_breach) if len(peaks) >= 1 else pd.DataFrame(
    columns=["peak_timestamp", "cmdb_id", "kpi_name", "global_P95", "first_breach_timestamp", "max_value_in_window", "severity_ratio"]
)
peak2_top_pairs = top_pairs_for_peak(peaks[1], pair_first_breach) if len(peaks) >= 2 else pd.DataFrame(
    columns=["peak_timestamp", "cmdb_id", "kpi_name", "global_P95", "first_breach_timestamp", "max_value_in_window", "severity_ratio"]
)

# Store variables in kernel (they are assigned above). Return compact results:
peak1_top_pairs, peak2_top_pairs

```
Out[7]:
```
Summary of top breached (cmdb_id, kpi_name) pairs at the two peak minutes:

- I saved the per-peak top-20 lists in the kernel as peak1_top_pairs and peak2_top_pairs (one DataFrame per peak timestamp).

- Peak 1: 2022-03-20 21:30:00 CST+0800
  - Top offending pairs include many cartservice-0 container metrics and several node/service metrics.
  - Highest severity ratios at this minute:
    - cartservice-0: container_fs_writes./dev/vda — severity 11.0
    - node-3: system.net.tcp.retrans_segs — severity ~8.45
    - cartservice-0: container_fs_writes_MB./dev/vda — severity ~8.43
    - node-1 and node-5: system.cpu.iowait — severity ~6.8 and 6.5
    - Several cartservice-0 CPU/memory/thread KPIs with severity 1.4–3.8
  - Implicated components (high-count/sample list): cartservice-0, node-3, node-1, node-4, recommendationservice-0, emailservice2-0, shippingservice-0, etc.

- Peak 2: 2022-03-20 21:31:00 CST+0800
  - Node-level IO and network metrics dominate the top severities.
  - Highest severity ratios at this minute:
    - node-1: system.io.rkb_s — severity ~19.16
    - node-1: system.io.util — severity ~14.11
    - paymentservice-2: container_network_receive_MB... — severity ~11.14
    - frontend-2: istio request-related trace/mesh metric — severity ~6.61
    - node-1 various network/IO metrics with severity 2–6 range
  - Implicated components: node-1 (multiple heavy IO/network KPIs), paymentservice-2, frontend-2, other nodes.

- Interpretation (straightforward):
  - The 21:30 minute shows strong container-level issues (notably cartservice-0) and some node CPU/iowait spikes.
  - The 21:31 minute is dominated by severe node-level IO/network anomalies on node-1 (very high IO and network metrics) plus a high-severity network receive anomaly on paymentservice-2.
  - Both minutes include high-severity anomalies, but the 21:31 minute shows the single largest severity ratios (node-1 IO metrics).

- Next actions you can take quickly:
  - Inspect node-1 (disk IO, network) around 21:31 and cartservice-0 container metrics around 21:30.
  - Review container/process logs for cartservice-0 and paymentservice-2, and system logs on node-1 for I/O errors or saturations.

If you want, I can export the top pairs for a specific component or show the full peak1_top_pairs / peak2_top_pairs DataFrames (up to 20 rows) for closer inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(                  peak_timestamp                  cmdb_id                                           kpi_name   global_P95        first_breach_timestamp  max_value_in_window  severity_ratio
0   2022-03-20 21:30:00 CST+0800            cartservice-0      container.node-6.container_fs_writes./dev/vda     1.500000  2022-03-20 21:30:00 CST+0800            16.500000       11.000000
1   2022-03-20 21:30:00 CST+0800                   node-3                        system.net.tcp.retrans_segs     0.200000  2022-03-20 21:30:00 CST+0800             1.690000        8.450000
2   2022-03-20 21:30:00 CST+0800            cartservice-0   container.node-6.container_fs_writes_MB./dev/vda     0.013672  2022-03-20 21:30:00 CST+0800             0.115234        8.428571
3   2022-03-20 21:30:00 CST+0800                   node-1                                  system.cpu.iowait     2.220000  2022-03-20 21:30:00 CST+0800            15.110000        6.806306
4   2022-03-20 21:30:00 CST+0800                   node-5                                  system.cpu.iowait     0.020000  2022-03-20 21:30:00 CST+0800             0.130000        6.500000
5   2022-03-20 21:30:00 CST+0800  recommendationservice-0  container.node-6.container_network_receive_MB....     0.137578  2022-03-20 21:30:00 CST+0800             0.685697        4.984043
6   2022-03-20 21:30:00 CST+0800            cartservice-0  container.node-6.container_cpu_cfs_throttled_s...     0.470722  2022-03-20 21:30:00 CST+0800             1.810134        3.845443
7   2022-03-20 21:30:00 CST+0800            cartservice-0  container.node-6.container_cpu_cfs_throttled_p...     5.000000  2022-03-20 21:30:00 CST+0800            12.000000        2.400000
8   2022-03-20 21:30:00 CST+0800        shippingservice-0  mesh.source.shippingservice.jaeger-collector.i...     0.001050  2022-03-20 21:30:00 CST+0800             0.002075        1.976190
9   2022-03-20 21:30:00 CST+0800                   node-4                               system.cpu.pct_usage     9.350000  2022-03-20 21:30:00 CST+0800            16.560000        1.771123
10  2022-03-20 21:30:00 CST+0800            cartservice-0        container.node-6.container_cpu_user_seconds     0.570000  2022-03-20 21:30:00 CST+0800             1.005000        1.763158
11  2022-03-20 21:30:00 CST+0800                   node-4                                  system.cpu.iowait     4.730000  2022-03-20 21:30:00 CST+0800             8.200000        1.733615
12  2022-03-20 21:30:00 CST+0800            cartservice-0  container.node-6.container_memory_failures.con...  4224.000000  2022-03-20 21:30:00 CST+0800          6963.000000        1.648438
13  2022-03-20 21:30:00 CST+0800            cartservice-0  container.node-6.container_memory_failures.hie...  4224.000000  2022-03-20 21:30:00 CST+0800          6963.000000        1.648438
14  2022-03-20 21:30:00 CST+0800            cartservice-0       container.node-6.container_cpu_usage_seconds     0.732883  2022-03-20 21:30:00 CST+0800             1.200796        1.638455
15  2022-03-20 21:30:00 CST+0800        shippingservice-0  mesh.destination.frontend.shippingservice.isti...     0.001050  2022-03-20 21:30:00 CST+0800             0.001575        1.500000
16  2022-03-20 21:30:00 CST+0800            cartservice-0                 container.node-6.container_threads    31.500000  2022-03-20 21:30:00 CST+0800            46.000000        1.460317
17  2022-03-20 21:30:00 CST+0800            cartservice-0      container.node-6.container_cpu_system_seconds     0.125000  2022-03-20 21:30:00 CST+0800             0.180000        1.440000
18  2022-03-20 21:30:00 CST+0800                   node-3                                 system.io.avg_q_sz     0.050000  2022-03-20 21:30:00 CST+0800             0.070000        1.400000
19  2022-03-20 21:30:00 CST+0800          emailservice2-0  container.node-6.container_network_transmit_pa...   243.000000  2022-03-20 21:30:00 CST+0800           328.500000        1.351852,                   peak_timestamp                  cmdb_id                                           kpi_name    global_P95        first_breach_timestamp  max_value_in_window  severity_ratio
0   2022-03-20 21:31:00 CST+0800                   node-1                                    system.io.rkb_s  6.220000e+02  2022-03-20 21:31:00 CST+0800         1.191600e+04       19.157556
1   2022-03-20 21:31:00 CST+0800                   node-1                                     system.io.util  3.300000e+00  2022-03-20 21:31:00 CST+0800         4.655000e+01       14.106061
2   2022-03-20 21:31:00 CST+0800         paymentservice-2  container.node-6.container_network_receive_MB....  2.842808e-02  2022-03-20 21:31:00 CST+0800         3.168077e-01       11.144185
3   2022-03-20 21:31:00 CST+0800               frontend-2  mesh.source.frontend.checkoutservice.istio_req...  3.087500e-01  2022-03-20 21:31:00 CST+0800         2.041000e+00        6.610526
4   2022-03-20 21:31:00 CST+0800                   node-1                                      system.io.r_s  3.250000e+01  2022-03-20 21:31:00 CST+0800         1.830000e+02        5.630769
5   2022-03-20 21:31:00 CST+0800                   node-1                                  system.io.r_await  5.700000e-01  2022-03-20 21:31:00 CST+0800         3.110000e+00        5.456140
6   2022-03-20 21:31:00 CST+0800                   node-1                            system.net.tcp.out_segs  8.488770e+03  2022-03-20 21:31:00 CST+0800         2.444060e+04        2.879169
7   2022-03-20 21:31:00 CST+0800                   node-1                              system.net.bytes_sent  1.463143e+06  2022-03-20 21:31:00 CST+0800         3.795882e+06        2.594334
8   2022-03-20 21:31:00 CST+0800                   node-2                        system.net.tcp.retrans_segs  1.700000e-01  2022-03-20 21:31:00 CST+0800         3.900000e-01        2.294118
9   2022-03-20 21:31:00 CST+0800                   node-1                              system.net.bytes_rcvd  2.743728e+06  2022-03-20 21:31:00 CST+0800         6.143480e+06        2.239099
10  2022-03-20 21:31:00 CST+0800                   node-1                        system.net.packets_in.count  2.055160e+03  2022-03-20 21:31:00 CST+0800         4.355580e+03        2.119339
11  2022-03-20 21:31:00 CST+0800                   node-1                             system.net.tcp.in_segs  3.163170e+03  2022-03-20 21:31:00 CST+0800         6.581370e+03        2.080625
12  2022-03-20 21:31:00 CST+0800  recommendationservice-2  container.node-6.container_network_receive_MB....  3.628201e-01  2022-03-20 21:31:00 CST+0800         6.967340e-01        1.920329
13  2022-03-20 21:31:00 CST+0800                   node-5                                  system.io.r_await  4.600000e-01  2022-03-20 21:31:00 CST+0800         8.800000e-01        1.913043
14  2022-03-20 21:31:00 CST+0800                   node-4                                      system.load.1  1.670000e+00  2022-03-20 21:31:00 CST+0800         2.820000e+00        1.688623
15  2022-03-20 21:31:00 CST+0800  productcatalogservice-1  mesh.destination.recommendationservice.product...  1.575000e-03  2022-03-20 21:31:00 CST+0800         2.625000e-03        1.666667
16  2022-03-20 21:31:00 CST+0800              adservice-0  mesh.source.adservice.jaeger-collector.istio_r...  4.625000e-03  2022-03-20 21:31:00 CST+0800         7.150000e-03        1.545946
17  2022-03-20 21:31:00 CST+0800        shippingservice-1      container.node-6.container_cpu_system_seconds  1.000000e-02  2022-03-20 21:31:00 CST+0800         1.500000e-02        1.500000
18  2022-03-20 21:31:00 CST+0800                   node-1                               system.cpu.pct_usage  3.324000e+01  2022-03-20 21:31:00 CST+0800         4.757000e+01        1.431107
19  2022-03-20 21:31:00 CST+0800               frontend-2  mesh.destination.unknown.frontend.istio_reques...  3.927750e+00  2022-03-20 21:31:00 CST+0800         5.476750e+00        1.394373)```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

tz = pytz.timezone("Asia/Shanghai")

# Ensure log_df is loaded in kernel and has timestamp_dt in UTC+8
if "log_df" not in globals():
    raise RuntimeError("Kernel variable 'log_df' not found.")

if "timestamp_dt" not in log_df.columns:
    log_df["timestamp_dt"] = pd.to_datetime(log_df["timestamp"], unit="s", utc=True).dt.tz_convert(tz)
else:
    # normalize/ensure timezone-aware Asia/Shanghai
    log_df["timestamp_dt"] = pd.to_datetime(log_df["timestamp_dt"], utc=True).dt.tz_convert(tz)

# Define window (inclusive both ends)
start = pd.Timestamp("2022-03-20 21:29:00").tz_localize(tz)
end = pd.Timestamp("2022-03-20 21:32:00").tz_localize(tz)

# Filter window
logs_window = log_df[(log_df["timestamp_dt"] >= start) & (log_df["timestamp_dt"] <= end)].copy()

# Target components
targets = ["cartservice-0", "paymentservice-2", "node-1", "node-3", "node-5"]

# 1) All rows for target cmdb_ids with requested columns, limit returned rows to 200 for display
logs_peak_window = logs_window[logs_window["cmdb_id"].isin(targets)].copy()
# create formatted timestamp column for display
logs_peak_window["timestamp"] = logs_peak_window["timestamp_dt"].dt.strftime("%Y-%m-%d %H:%M:%S %Z%z")
logs_peak_window = logs_peak_window[["timestamp", "cmdb_id", "log_name", "value"]].reset_index(drop=True)

# 2) Aggregate table: group by cmdb_id and log_name, count rows and sum numeric values
# convert value to numeric where possible
logs_peak_window["value_num"] = pd.to_numeric(logs_peak_window["value"], errors="coerce")
logs_peak_agg = logs_peak_window.groupby(["cmdb_id", "log_name"], as_index=False).agg(
    row_count = ("value", "size"),
    value_sum = ("value_num", "sum")
)
# sort by count desc and limit to top 20 groups
logs_peak_agg = logs_peak_agg.sort_values(by=["row_count", "value_sum"], ascending=[False, False]).reset_index(drop=True)
logs_peak_agg = logs_peak_agg.head(20)

# Store variables in kernel as requested (already assigned): logs_peak_window, logs_peak_agg
# Return compact views: up to 200 rows for raw rows and up to 20 rows for aggregates
logs_peak_window.head(200), logs_peak_agg.head(20)

```
Out[8]:
```
```python
# Build a plain-English summary from the already-filtered log results in kernel:
# Uses: logs_peak_window, logs_peak_agg

# Defensive checks
missing = [v for v in ("logs_peak_window", "logs_peak_agg") if v not in globals()]
if missing:
    summary = f"Required kernel variables missing: {', '.join(missing)}"
else:
    lw = logs_peak_window.copy()
    la = logs_peak_agg.copy()

    # Extract observations for cartservice-0 and paymentservice-2
    def rows_for(cmdb):
        sub = lw[lw["cmdb_id"] == cmdb]
        if sub.empty:
            return "no log rows in window"
        # summarize row_count and error_count if present
        rc = sub[sub["log_name"] == "log.row_count"]["value"].astype(float).tolist()
        ec = sub[sub["log_name"] == "log.error_count"]["value"].astype(float).tolist()
        parts = []
        if rc:
            parts.append("log.row_count per-minute = " + ", ".join(str(int(x)) for x in rc))
        if ec:
            parts.append("log.error_count per-minute = " + ", ".join(str(int(x)) for x in ec))
        return "; ".join(parts) if parts else "rows present, no numeric summaries"

    cart_summary = rows_for("cartservice-0")
    pay_summary = rows_for("paymentservice-2")
    node1_summary = rows_for("node-1")
    node3_summary = rows_for("node-3")
    node5_summary = rows_for("node-5")

    # Aggregates summary
    agg_lines = []
    for _, r in la.iterrows():
        agg_lines.append(f"{r['cmdb_id']} | {r['log_name']} → count={int(r['row_count'])}, sum(value)={float(r['value_sum']):.1f}")

    summary_lines = [
        "Log inspection (2022-03-20 21:29–21:32 CST+0800) for selected components:",
        "",
        f"- cartservice-0: {cart_summary}.",
        f"- paymentservice-2: {pay_summary}.",
        f"- node-1: {node1_summary}.",
        f"- node-3: {node3_summary}.",
        f"- node-5: {node5_summary}.",
        "",
        "Aggregate counts in the window (top groups):",
    ]
    if agg_lines:
        summary_lines.extend("  - " + l for l in agg_lines)
    else:
        summary_lines.append("  - No aggregate log groups found.")

    summary_lines.extend([
        "",
        "Interpretation:",
        "- cartservice-0 shows very high per-minute log.row_count at the peak minutes (619, 531, 591) but log.error_count = 0 → heavy traffic but no logged errors in these normalized log metrics.",
        "- paymentservice-2 shows low per-minute row_count (9, 8, 9) and log.error_count = 0 → low traffic, no logged errors.",
        "- No log entries for node-1, node-3, node-5 in log.csv for this window → log.csv does not corroborate node-level issues; investigate system logs/metrics for node-level anomalies (CPU/IO/network) seen in metrics.",
        "",
        "Stored kernel variables for follow-up: 'logs_peak_window' (raw rows) and 'logs_peak_agg' (aggregates)."
    ])

    summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                       timestamp           cmdb_id         log_name  value  value_num
0   2022-03-20 21:30:00 CST+0800     cartservice-0  log.error_count    0.0        0.0
1   2022-03-20 21:30:00 CST+0800     cartservice-0    log.row_count  619.0      619.0
2   2022-03-20 21:30:00 CST+0800  paymentservice-2  log.error_count    0.0        0.0
3   2022-03-20 21:30:00 CST+0800  paymentservice-2    log.row_count    9.0        9.0
4   2022-03-20 21:31:00 CST+0800     cartservice-0  log.error_count    0.0        0.0
5   2022-03-20 21:31:00 CST+0800     cartservice-0    log.row_count  531.0      531.0
6   2022-03-20 21:31:00 CST+0800  paymentservice-2  log.error_count    0.0        0.0
7   2022-03-20 21:31:00 CST+0800  paymentservice-2    log.row_count    8.0        8.0
8   2022-03-20 21:32:00 CST+0800     cartservice-0  log.error_count    0.0        0.0
9   2022-03-20 21:32:00 CST+0800     cartservice-0    log.row_count  591.0      591.0
10  2022-03-20 21:32:00 CST+0800  paymentservice-2  log.error_count    0.0        0.0
11  2022-03-20 21:32:00 CST+0800  paymentservice-2    log.row_count    9.0        9.0,             cmdb_id         log_name  row_count  value_sum
0     cartservice-0    log.row_count          3     1741.0
1  paymentservice-2    log.row_count          3       26.0
2     cartservice-0  log.error_count          3        0.0
3  paymentservice-2  log.error_count          3        0.0)```
```